# Bean Sprout Growth Experiment - Data Analysis
## The Effect of Colored Light on Mung Bean Sprout Growth
**CID: 06043088** | ELEC70126 - Internet of Things and Applications

This notebook performs data cleaning, exploratory analysis, time-series analytics, and statistical inference on the photoresistor sensor data collected from the IoT bean sprout growth experiment.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats, signal
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 150

IMAGE_DIR = '../images/'

# Google Sheets live data source
SHEET_ID = "1Vkc24L-VDzpiR6GrKL9sg7BJ5h5271bmTEASLvmzD9M"
SHEET_TAB = "data"
GSHEET_URL = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/gviz/tq?tqx=out:csv&sheet={SHEET_TAB}"

# Local fallback
LOCAL_CSV = '../data/experiment_result.csv'

## 1. Data Loading and Initial Inspection

In [ ]:
# Try Google Sheets first, fall back to local CSV
try:
    df = pd.read_csv(GSHEET_URL)
    print("Data loaded from Google Sheets (live)")
except Exception as e:
    print(f"Google Sheets unavailable ({e}), using local CSV")
    df = pd.read_csv(LOCAL_CSV)

# Handle both timestamp formats: "2026-03-05 2:38:27" (Sheets) and "05/03/2026 02:38" (CSV)
df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='mixed', dayfirst=True)
df = df.sort_values('Timestamp').reset_index(drop=True)

print(f"Dataset shape: {df.shape}")
print(f"Time range: {df['Timestamp'].min()} to {df['Timestamp'].max()}")
print(f"Duration: {df['Timestamp'].max() - df['Timestamp'].min()}")
print(f"Sampling interval (median): {df['Timestamp'].diff().median()}")
print()
df.describe()

In [ ]:
# Quick look at the raw data
print("First 5 rows:")
display(df.head())
print("\nLast 5 rows:")
display(df.tail())

## 2. Data Cleaning

### 2.1 Identifying Anomalies
Some data points show extreme simultaneous drops across all three photoresistor channels. These are caused by physical disturbances during watering (e.g., unintentionally shifting the sensor or touching the breadboard). We identify these as rows where **all three** sensor values change drastically at once.

In [ ]:
# Compute absolute differences between consecutive readings
diff_green = df['Green'].diff().abs()
diff_blue = df['Blue'].diff().abs()
diff_control = df['Control'].diff().abs()

# Anomaly: all three channels jump by more than 200 simultaneously
THRESHOLD = 200
anomaly_mask = (diff_green > THRESHOLD) & (diff_blue > THRESHOLD) & (diff_control > THRESHOLD)

# Also check for the recovery point (next row after anomaly)
anomaly_indices = df.index[anomaly_mask].tolist()
# Include the anomalous rows themselves
rows_to_remove = set()
for idx in anomaly_indices:
    rows_to_remove.add(idx)
    # Check if previous row is also anomalous (the disturbance point)
    if idx > 0:
        prev_vals = df.loc[idx-1, ['Green', 'Blue', 'Control']].values
        curr_vals = df.loc[idx, ['Green', 'Blue', 'Control']].values
        # If previous row values are way off from the trend, remove it too
        if idx - 1 not in anomaly_indices:
            rows_to_remove.add(idx - 1)

print(f"Anomalous rows detected: {len(rows_to_remove)}")
print("\nAnomalous data points:")
for idx in sorted(rows_to_remove):
    print(f"  Row {idx}: {df.loc[idx, 'Timestamp']} | G={df.loc[idx, 'Green']} B={df.loc[idx, 'Blue']} C={df.loc[idx, 'Control']}")

In [ ]:
# Visualise anomalies on raw data
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df['Timestamp'], df['Green'], 'g-', alpha=0.7, label='Green', linewidth=0.8)
ax.plot(df['Timestamp'], df['Blue'], 'b-', alpha=0.7, label='Blue', linewidth=0.8)
ax.plot(df['Timestamp'], df['Control'], 'k-', alpha=0.7, label='Control', linewidth=0.8)

# Mark anomalies
for idx in sorted(rows_to_remove):
    ax.axvline(df.loc[idx, 'Timestamp'], color='red', alpha=0.5, linestyle='--', linewidth=0.8)

ax.set_xlabel('Time')
ax.set_ylabel('Photoresistor Reading (ADC)')
ax.set_title('Raw Sensor Data with Anomalies Highlighted (Red Lines)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.tight_layout()
plt.savefig(IMAGE_DIR + 'raw_data_with_anomalies.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Remove anomalies and interpolate
df_clean = df.copy()
for idx in rows_to_remove:
    df_clean.loc[idx, ['Green', 'Blue', 'Control']] = np.nan

# Interpolate missing values
df_clean[['Green', 'Blue', 'Control']] = df_clean[['Green', 'Blue', 'Control']].interpolate(method='linear')

print(f"Cleaned dataset: {df_clean.shape[0]} rows ({len(rows_to_remove)} anomalous points interpolated)")
print(f"Remaining NaNs: {df_clean[['Green', 'Blue', 'Control']].isna().sum().sum()}")

## 3. Exploratory Data Analysis

### 3.1 Growth Curves (Cleaned Data)

The photoresistor reading **increases** as the plant grows, because the growing sprout obstructs more of the light path between the LED and the sensor. A higher reading indicates more obstruction, hence more growth.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Plot 1: Absolute sensor readings
ax1 = axes[0]
ax1.plot(df_clean['Timestamp'], df_clean['Green'], color='#2ca02c', label='Green Light', linewidth=1.2)
ax1.plot(df_clean['Timestamp'], df_clean['Blue'], color='#1f77b4', label='Blue Light', linewidth=1.2)
ax1.plot(df_clean['Timestamp'], df_clean['Control'], color='#333333', label='Control (Dark)', linewidth=1.2)
ax1.set_ylabel('Photoresistor Reading (ADC)')
ax1.set_title('Bean Sprout Growth Curves — Photoresistor Readings Over Time')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)
ax1.axhline(y=4095, color='red', linestyle=':', alpha=0.5, label='Sensor Saturation (4095)')
ax1.legend(loc='upper left')

# Plot 2: Normalised growth (relative change from baseline)
ax2 = axes[1]
baseline_green = df_clean['Green'].iloc[:10].mean()
baseline_blue = df_clean['Blue'].iloc[:10].mean()
baseline_control = df_clean['Control'].iloc[:10].mean()

ax2.plot(df_clean['Timestamp'], (df_clean['Green'] - baseline_green), color='#2ca02c', label=f'Green (baseline={baseline_green:.0f})', linewidth=1.2)
ax2.plot(df_clean['Timestamp'], (df_clean['Blue'] - baseline_blue), color='#1f77b4', label=f'Blue (baseline={baseline_blue:.0f})', linewidth=1.2)
ax2.plot(df_clean['Timestamp'], (df_clean['Control'] - baseline_control), color='#333333', label=f'Control (baseline={baseline_control:.0f})', linewidth=1.2)
ax2.set_ylabel('Change from Baseline (ADC units)')
ax2.set_xlabel('Time')
ax2.set_title('Relative Growth — Change from Initial Baseline Reading')
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3)
ax2.axhline(y=0, color='gray', linestyle='-', alpha=0.3)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d\n%H:%M'))

plt.tight_layout()
plt.savefig(IMAGE_DIR + 'growth_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.2 Environmental Conditions

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ax1.plot(df_clean['Timestamp'], df_clean['Temp(C)'], color='#d62728', linewidth=1)
ax1.set_ylabel('Temperature (°C)')
ax1.set_title('Environmental Conditions During Experiment')
ax1.grid(True, alpha=0.3)
ax1.fill_between(df_clean['Timestamp'], df_clean['Temp(C)'].min(), df_clean['Temp(C)'], alpha=0.15, color='red')

ax2.plot(df_clean['Timestamp'], df_clean['Humidity(%)'], color='#17becf', linewidth=1)
ax2.set_ylabel('Humidity (%)')
ax2.set_xlabel('Time')
ax2.grid(True, alpha=0.3)
ax2.fill_between(df_clean['Timestamp'], df_clean['Humidity(%)'].min(), df_clean['Humidity(%)'], alpha=0.15, color='cyan')

for ax in [ax1, ax2]:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d\n%H:%M'))

plt.tight_layout()
plt.savefig(IMAGE_DIR + 'environmental_conditions.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Temperature — Mean: {df_clean['Temp(C)'].mean():.1f}°C, Min: {df_clean['Temp(C)'].min():.1f}°C, Max: {df_clean['Temp(C)'].max():.1f}°C")
print(f"Humidity    — Mean: {df_clean['Humidity(%)'].mean():.1f}%, Min: {df_clean['Humidity(%)'].min():.1f}%, Max: {df_clean['Humidity(%)'].max():.1f}%")

### 3.3 Growth Rate Analysis
We compute hourly growth rates using a rolling window to smooth the 15-minute sampling noise.

In [ ]:
# Compute hourly growth rate (4 samples = 1 hour)
window = 4  # 4 x 15min = 1 hour
df_clean['Green_rate'] = df_clean['Green'].diff(window) / window
df_clean['Blue_rate'] = df_clean['Blue'].diff(window) / window
df_clean['Control_rate'] = df_clean['Control'].diff(window) / window

# Smooth with rolling average
smooth_window = 8  # 2-hour smoothing
df_clean['Green_rate_smooth'] = df_clean['Green_rate'].rolling(smooth_window, center=True).mean()
df_clean['Blue_rate_smooth'] = df_clean['Blue_rate'].rolling(smooth_window, center=True).mean()
df_clean['Control_rate_smooth'] = df_clean['Control_rate'].rolling(smooth_window, center=True).mean()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df_clean['Timestamp'], df_clean['Green_rate_smooth'], color='#2ca02c', label='Green', linewidth=1.2)
ax.plot(df_clean['Timestamp'], df_clean['Blue_rate_smooth'], color='#1f77b4', label='Blue', linewidth=1.2)
ax.plot(df_clean['Timestamp'], df_clean['Control_rate_smooth'], color='#333333', label='Control', linewidth=1.2)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_ylabel('Growth Rate (ADC units / 15 min)')
ax.set_xlabel('Time')
ax.set_title('Smoothed Growth Rate Over Time (2-Hour Rolling Average)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d\n%H:%M'))
plt.tight_layout()
plt.savefig(IMAGE_DIR + 'growth_rate.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.4 Plant Movement / Oscillation Analysis (Control Group)

A key unexpected finding: the **control group** sensor data shows periodic oscillations (up-and-down movement). Since the control chamber is dark, the sprouts exhibit **phototropic movement** — likely bending toward any small light leak from the adjacent blue chamber. This oscillatory pattern reveals the plant's movement behavior over time.

In [ ]:
# Detrend the control signal to isolate oscillation
# Use a large rolling mean as the trend
trend_window = 48  # 12-hour trend
df_clean['Control_trend'] = df_clean['Control'].rolling(trend_window, center=True).mean()
df_clean['Control_detrended'] = df_clean['Control'] - df_clean['Control_trend']

# Also detrend green and blue for comparison
df_clean['Green_trend'] = df_clean['Green'].rolling(trend_window, center=True).mean()
df_clean['Green_detrended'] = df_clean['Green'] - df_clean['Green_trend']
df_clean['Blue_trend'] = df_clean['Blue'].rolling(trend_window, center=True).mean()
df_clean['Blue_detrended'] = df_clean['Blue'] - df_clean['Blue_trend']

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for ax, col, color, name in zip(axes, 
    ['Green_detrended', 'Blue_detrended', 'Control_detrended'],
    ['#2ca02c', '#1f77b4', '#333333'],
    ['Green Light', 'Blue Light', 'Control (Dark)']):
    ax.plot(df_clean['Timestamp'], df_clean[col], color=color, linewidth=0.8)
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax.set_ylabel('Detrended ADC')
    ax.set_title(f'{name} — Detrended Signal (Oscillation / Plant Movement)')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time')
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d\n%H:%M'))
plt.tight_layout()
plt.savefig(IMAGE_DIR + 'plant_movement_oscillation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Frequency analysis using FFT on control detrended signal
control_detrended = df_clean['Control_detrended'].dropna().values
n = len(control_detrended)
sampling_interval_hours = 0.25  # 15 minutes = 0.25 hours

fft_vals = np.fft.rfft(control_detrended)
fft_freq = np.fft.rfftfreq(n, d=sampling_interval_hours)  # in cycles per hour
fft_power = np.abs(fft_vals) ** 2

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(1/fft_freq[1:], fft_power[1:] / fft_power[1:].max(), color='#333333', linewidth=0.8)
ax.set_xlabel('Period (hours)')
ax.set_ylabel('Normalised Power')
ax.set_title('Control Group — Frequency Spectrum (FFT) of Detrended Signal')
ax.set_xlim(0, 48)
ax.grid(True, alpha=0.3)
ax.axvline(x=24, color='red', linestyle='--', alpha=0.5, label='24-hour period')
ax.axvline(x=12, color='orange', linestyle='--', alpha=0.5, label='12-hour period')
ax.legend()
plt.tight_layout()
plt.savefig(IMAGE_DIR + 'fft_control_movement.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.5 Cross-Correlation Between Sensor Channels

In [ ]:
# Cross-correlation between channels
from scipy.signal import correlate

valid = df_clean[['Green', 'Blue', 'Control', 'Temp(C)', 'Humidity(%)']].dropna()
corr_matrix = valid.corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_yticks(range(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=45, ha='right')
ax.set_yticklabels(corr_matrix.columns)

for i in range(len(corr_matrix)):
    for j in range(len(corr_matrix)):
        ax.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}', ha='center', va='center', fontsize=10)

plt.colorbar(im, ax=ax, label='Pearson Correlation')
ax.set_title('Cross-Correlation Matrix — All Sensor Channels')
plt.tight_layout()
plt.savefig(IMAGE_DIR + 'correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.6 Daily Summary Statistics

In [ ]:
df_clean['Date'] = df_clean['Timestamp'].dt.date
daily = df_clean.groupby('Date').agg({
    'Green': ['mean', 'min', 'max', 'std'],
    'Blue': ['mean', 'min', 'max', 'std'],
    'Control': ['mean', 'min', 'max', 'std'],
    'Temp(C)': ['mean', 'min', 'max'],
    'Humidity(%)': ['mean', 'min', 'max']
}).round(1)

print("Daily Summary Statistics:")
display(daily)

In [ ]:
# Daily growth (difference between first and last reading each day)
daily_growth = df_clean.groupby('Date').agg(
    Green_start=('Green', 'first'),
    Green_end=('Green', 'last'),
    Blue_start=('Blue', 'first'),
    Blue_end=('Blue', 'last'),
    Control_start=('Control', 'first'),
    Control_end=('Control', 'last')
)
daily_growth['Green_daily'] = daily_growth['Green_end'] - daily_growth['Green_start']
daily_growth['Blue_daily'] = daily_growth['Blue_end'] - daily_growth['Blue_start']
daily_growth['Control_daily'] = daily_growth['Control_end'] - daily_growth['Control_start']

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(daily_growth))
width = 0.25
ax.bar(x - width, daily_growth['Green_daily'], width, label='Green', color='#2ca02c', alpha=0.8)
ax.bar(x, daily_growth['Blue_daily'], width, label='Blue', color='#1f77b4', alpha=0.8)
ax.bar(x + width, daily_growth['Control_daily'], width, label='Control', color='#555555', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([str(d) for d in daily_growth.index], rotation=30)
ax.set_ylabel('Daily ADC Change')
ax.set_title('Daily Growth (ADC Change per Day)')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
plt.tight_layout()
plt.savefig(IMAGE_DIR + 'daily_growth.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.7 Cumulative Growth Comparison

In [ ]:
# Total growth from start to end
total_green = df_clean['Green'].iloc[-1] - df_clean['Green'].iloc[0]
total_blue = df_clean['Blue'].iloc[-1] - df_clean['Blue'].iloc[0]
total_control = df_clean['Control'].iloc[-1] - df_clean['Control'].iloc[0]

# Note: Blue saturated at 4095, so actual growth is higher
print("=== Total Growth (ADC units from start to end) ===")
print(f"Green:   {total_green:+.0f}  (Start: {df_clean['Green'].iloc[0]:.0f}, End: {df_clean['Green'].iloc[-1]:.0f})")
print(f"Blue:    {total_blue:+.0f}  (Start: {df_clean['Blue'].iloc[0]:.0f}, End: {df_clean['Blue'].iloc[-1]:.0f}) ⚠️ Saturated at 4095")
print(f"Control: {total_control:+.0f}  (Start: {df_clean['Control'].iloc[0]:.0f}, End: {df_clean['Control'].iloc[-1]:.0f})")

fig, ax = plt.subplots(figsize=(8, 5))
groups = ['Green Light', 'Blue Light\n(Saturated)', 'Control\n(Dark)']
values = [total_green, total_blue, total_control]
colors = ['#2ca02c', '#1f77b4', '#555555']
bars = ax.bar(groups, values, color=colors, alpha=0.85, edgecolor='white', linewidth=2)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10, f'{val:+.0f}', 
            ha='center', va='bottom', fontweight='bold', fontsize=12)

ax.set_ylabel('Total ADC Change')
ax.set_title('Cumulative Growth Comparison (Start to End)')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(IMAGE_DIR + 'cumulative_growth.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Statistical Analysis

### 4.1 Growth Rate Comparison (t-tests)

In [ ]:
# Compare growth rates between groups (using non-null rates)
rates = df_clean[['Green_rate', 'Blue_rate', 'Control_rate']].dropna()

# Welch's t-test: Green vs Blue
t_gb, p_gb = stats.ttest_ind(rates['Green_rate'], rates['Blue_rate'], equal_var=False)
t_gc, p_gc = stats.ttest_ind(rates['Green_rate'], rates['Control_rate'], equal_var=False)
t_bc, p_bc = stats.ttest_ind(rates['Blue_rate'], rates['Control_rate'], equal_var=False)

print("=== Welch's t-test on Hourly Growth Rates ===")
print(f"Green vs Blue:    t={t_gb:.3f}, p={p_gb:.4f} {'***' if p_gb < 0.001 else '**' if p_gb < 0.01 else '*' if p_gb < 0.05 else 'ns'}")
print(f"Green vs Control: t={t_gc:.3f}, p={p_gc:.4f} {'***' if p_gc < 0.001 else '**' if p_gc < 0.01 else '*' if p_gc < 0.05 else 'ns'}")
print(f"Blue vs Control:  t={t_bc:.3f}, p={p_bc:.4f} {'***' if p_bc < 0.001 else '**' if p_bc < 0.01 else '*' if p_bc < 0.05 else 'ns'}")

print(f"\nMean growth rates (ADC/15min):")
print(f"  Green:   {rates['Green_rate'].mean():.3f} ± {rates['Green_rate'].std():.3f}")
print(f"  Blue:    {rates['Blue_rate'].mean():.3f} ± {rates['Blue_rate'].std():.3f}")
print(f"  Control: {rates['Control_rate'].mean():.3f} ± {rates['Control_rate'].std():.3f}")

### 4.2 Correlation Between Environmental Factors and Growth

In [ ]:
# Correlation between temp/humidity and growth rates
env_corr = df_clean[['Green_rate', 'Blue_rate', 'Control_rate', 'Temp(C)', 'Humidity(%)']].dropna().corr()
print("Correlation between environmental factors and growth rates:")
print(env_corr.loc[['Temp(C)', 'Humidity(%)'], ['Green_rate', 'Blue_rate', 'Control_rate']].round(3))

### 4.3 Blue Channel Saturation Analysis

The blue channel reached the ADC maximum (4095) early in the experiment, meaning the sensor was fully saturated. This indicates the blue-light group had the fastest and most vigorous growth.

In [ ]:
# Find when blue first saturated
blue_saturated = df_clean[df_clean['Blue'] >= 4095]
if len(blue_saturated) > 0:
    first_sat = blue_saturated['Timestamp'].iloc[0]
    start_time = df_clean['Timestamp'].iloc[0]
    hours_to_saturation = (first_sat - start_time).total_seconds() / 3600
    pct_saturated = len(blue_saturated) / len(df_clean) * 100
    print(f"Blue channel first saturated at: {first_sat}")
    print(f"Time to saturation: {hours_to_saturation:.1f} hours ({hours_to_saturation/24:.1f} days)")
    print(f"Percentage of readings at saturation: {pct_saturated:.1f}%")
else:
    print("Blue channel never reached saturation")

## 5. Key Findings Summary

1. **Blue light accelerates growth the most**: The blue-light group grew so rapidly that the photoresistor sensor saturated at 4095 (ADC max) within ~3 days, indicating the fastest and most vigorous growth.

2. **Green light has moderate effect**: The green-light group showed steady growth but significantly less than the blue-light group, reaching ~3528 by the end of the experiment (from ~3479 baseline).

3. **Control group reveals plant movement**: The dark (control) group exhibited fascinating oscillatory behavior — the sensor readings moved up and down periodically. This indicates **phototropic movement** (nutation), likely the sprouts bending toward faint light leaks from the adjacent blue chamber.

4. **Environmental conditions were stable**: Temperature ranged from ~24.6–27.1°C and humidity from ~54–62%, providing consistent growing conditions across all groups.

5. **Data anomalies from physical disturbance**: A few data points showed simultaneous drops across all channels, caused by physical contact during watering. These were successfully cleaned via interpolation.

In [ ]:
print("Analysis complete. All figures saved to ../images/")